In [ ]:
from __future__ import annotations
import pandas as pd, numpy as np
from pathlib import Path
from typing import Optional, Literal
from pandas.api.types import is_numeric_dtype, is_bool_dtype
import json, warnings

# Evidently
from evidently import Report
from evidently.presets import DataDriftPreset
from evidently.metrics import ValueDrift, DriftedColumnsCount
from evidently import DataDefinition, Dataset

warnings.filterwarnings("ignore")


from funciones_analisis import ref_decay_prefix_mass, ref_golden, ref_seasonal

In [ ]:
plant_files = {
    "planta1": Path("../df_procesados/df_planta_1.csv"),
    "planta2": Path("../df_procesados/df_planta_2.csv"),
    "planta3": Path("../df_procesados/df_planta_3.csv"),
}
flag_files = {
    "planta1": Path("../df_procesados/flags_p1.csv"),
    "planta2": Path("../df_procesados/flags_p2.csv"),
    "planta3": Path("../df_procesados/flags_p3.csv")
}

output_dir_base = Path("../reportes_D3/newnew")
output_dir_base.mkdir(parents=True, exist_ok=True)

output_dirs = {plant: output_dir_base / plant for plant in plant_files}
for d in output_dirs.values():
    d.mkdir(parents=True, exist_ok=True)

output_dir = None  # Se asignará por planta en el ciclo principal
#---------------------------------------------------------------------------------#
# Ventanas por TIEMPO (no por número de filas)
CURRENT_WINDOW: str = "3D"          # tramo "actual": últimos 3 días (ajústalo)

CURRENT_BY_SAMPLES: bool = False     # False = usa CURRENT_WINDOW por tiempo (como hoy)
CURRENT_LAST_N: int = 1440            # últimos 1440 minutos (24 horas)
# Parámetros por estrategia de baseline (todas determinísticas)
BASELINE_STRATEGY: Literal["decay","golden","seasonal"] = "golden"
DECAY_HALF_LIFE_HOURS: int = 24*7   # half-life 1 semana
DECAY_WEIGHT_MASS: float = 0.95     # tomar el prefijo más reciente que acumule 95% de masa
GOLDEN_WIN: str = "30min"
GOLDEN_STEP: str = "10min"
GOLDEN_K: int = 40                  # K ventanas históricas "estables"
SEASONAL_WEEKS_BACK: int = 12       # semanas hacia atrás para matching hora/semana



#---------------------------------------------------------------------------------#
# Tamaño sample, puede ser None(se toman todos los datos)
RESAMPLE: Optional[str] = None

# Como se calcula valor sample
RESAMPLE_AGG: Literal["mean","median"] = "mean"
#---------------------------------------------------------------------------------#
# Columnas con las que no se quiere trabajar
EXCLUDE_COLUMNS: list[str] = ['pH Ecualizador 2 (Tk 250m3)', "Conductividad DAF", "Temperatura DAF",
                               'pH entrada a Ecualizador 1', "Flujo Aire Reactor 1", "OD Reactor 1"]
#---------------------------------------------------------------------------------#
# Columna prioritaria (KPI)
KPI_ENABLED = False
KPI_COL     = ""
KPI_IN_FILENAME = True
#---------------------------------------------------------------------------------#
# Método estadístico Evidently en columnas numéricas
NUM_METHOD: Literal["auto","ks","wasserstein","psi","anderson","cramer","mannwhitney"] = "auto"

In [ ]:
def strip_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """Si existe is_outlier, elimina filas marcadas como outlier (True/1/'true')."""
    if "is_outlier" not in df.columns:
        return df
    s = df["is_outlier"]
    # considerar 1/True/'true'/'True' como outlier
    mask = ~(s.astype(str).str.lower().isin(["1","true","t","yes","y"]))
    return df.loc[mask].drop(columns=["is_outlier"])

def resample_mixed(df: pd.DataFrame, freq: str, agg: str) -> pd.DataFrame:
    """Resample para mixto: numéricas por mean/median, categóricas por moda."""
    if df.empty: return df
    num = df.select_dtypes(include="number")
    other_cols = [c for c in df.columns if c not in num.columns]
    if agg == "median":
        num_rs = num.resample(freq).median()
    else:
        num_rs = num.resample(freq).mean()
    if other_cols:
        # moda por bloque (si hay empate, toma la primera)
        def _mode(s: pd.Series):
            s = s.dropna()
            if s.empty: return np.nan
            counts = s.value_counts()
            return counts.index[0]
        other = df[other_cols]
        other_rs = other.resample(freq).agg(_mode)
        out = pd.concat([num_rs, other_rs], axis=1)
    else:
        out = num_rs
    # reordenar columnas como original
    out = out[[c for c in df.columns if c in out.columns]]
    return out

def build_types_keep_all(ref: pd.DataFrame, cur: pd.DataFrame, dt_col: str) -> tuple[list[str], list[str], list[str]]:
    """
    Devuelve (numeric_cols, categorical_cols, dropped_all_nan)
    Incluye TODAS las columnas comunes salvo:
      - dt_col, EXCLUDE_COLUMNS
      - columnas 100% NaN en ref y cur
    """
    common = [c for c in ref.columns.intersection(cur.columns) if c != dt_col and c not in EXCLUDE_COLUMNS]
    numeric_cols, categorical_cols, dropped_all_nan = [], [], []
    for c in common:
        r, k = ref[c], cur[c]
        if r.dropna().empty and k.dropna().empty:
            dropped_all_nan.append(c); continue
        if is_bool_dtype(r) or is_bool_dtype(k):
            categorical_cols.append(c)
        elif is_numeric_dtype(r) or is_numeric_dtype(k):
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)
    return numeric_cols, categorical_cols, dropped_all_nan

def window_starts(index: pd.DatetimeIndex, win: pd.Timedelta, step: pd.Timedelta):
    if len(index) == 0: return []
    t, tmax = index.min(), index.max()
    out = []
    while t + win <= tmax:
        out.append(t); t = t + step
    return out

In [ ]:
def ref_decay_prefix_mass(df_hist: pd.DataFrame, now: pd.Timestamp,
                          half_life_hours=24*7, target_mass=0.95) -> pd.DataFrame:
    """
    determinístico: calcula pesos w = exp(-Δt/τ), ordena por recencia,
    y toma el prefijo más reciente cuya masa acumulada >= target_mass.
    """
    if df_hist.empty: return df_hist
    tau = pd.Timedelta(hours=half_life_hours) / np.log(2)
    dt = (now - df_hist.index)
    w = np.exp(-dt / tau).astype(float)
    order = np.argsort(-df_hist.index.view("i8"))  # descendente por tiempo
    w_sorted = w.values[order]
    cum = np.cumsum(w_sorted) / w_sorted.sum()
    cut_idx = np.searchsorted(cum, target_mass, side="left")
    # tomar hasta cut_idx (inclusive)
    take_pos = order[: (cut_idx + 1)]
    sel = df_hist.iloc[np.sort(take_pos)]
    return sel

def ref_golden(df_hist: pd.DataFrame, win="30min", step="10min", k=40) -> pd.DataFrame:
    """Elige K ventanas históricas más 'estables' (score robusto)."""
    win_td, step_td = pd.to_timedelta(win), pd.to_timedelta(step)
    starts = window_starts(df_hist.index, win_td, step_td)
    if not starts: return df_hist.iloc[:0]
    rows = []
    for t0 in starts:
        t1 = t0 + win_td - pd.Timedelta(nanoseconds=1)
        sub = df_hist.loc[t0:t1]
        if len(sub) < 3: continue
        num = sub.select_dtypes(include="number")
        if num.shape[1] == 0: continue
        med = num.median()
        iqr = num.quantile(0.75) - num.quantile(0.25)
        rsd = (iqr / (med.abs() + 1e-12)).replace([np.inf, -np.inf], np.nan)
        score = rsd.median(skipna=True)
        rows.append((t0, t1, float(score)))
    if not rows: return df_hist.iloc[:0]
    stab = pd.DataFrame(rows, columns=["t0","t1","score"]).sort_values("score").head(k)
    parts = [df_hist.loc[t0:t1] for t0, t1, _ in stab.itertuples(index=False)]
    return pd.concat(parts, axis=0) if parts else df_hist.iloc[:0]

def ref_seasonal(df_hist: pd.DataFrame, current_end: pd.Timestamp, weeks_back=12) -> pd.DataFrame:
    """Referencia estacional: misma hora-del-día y día-de-semana, W semanas atrás."""
    if df_hist.empty: return df_hist.iloc[:0]
    slot = current_end.dayofweek * 24 + current_end.hour
    dw, hh = df_hist.index.dayofweek, df_hist.index.hour
    mask = (dw * 24 + hh) == slot
    hist = df_hist.loc[mask].loc[:current_end]
    if hist.empty: return df_hist.iloc[:0]
    start_lim = current_end - pd.Timedelta(weeks=weeks_back)
    return hist.loc[start_lim:]

In [ ]:
def extract_value_drift_table(report, snap=None) -> pd.DataFrame:
    """
    Devuelve una tabla por columna con: col, drifted, score, method, threshold.
    Soporta tanto Report como Snapshot según la versión de Evidently.
    """
    # 1) Obtenemos el dict del reporte de la forma que exista
    d = None
    if hasattr(report, "as_dict"):        # Evidently >=0.7 (usualmente en Report)
        d = report.as_dict()
    elif hasattr(report, "json"):         # Algunas versiones exponen .json() en Report
        d = json.loads(report.json())
    elif snap is not None and hasattr(snap, "json"):  # O en Snapshot
        d = json.loads(snap.json())
    else:
        raise RuntimeError("No se pudo serializar el Report/Snapshot a dict (as_dict/json no disponibles).")

    # 2) Parseamos métricas ValueDrift
    rows = []
    for m in d.get("metrics", []):
        if m.get("metric") == "ValueDrift":
            res = m.get("result", {}) or {}
            rows.append({
                "col":       res.get("column_name") or res.get("column"),
                "drifted":   res.get("drift_detected"),
                "score":     res.get("drift_score"),
                "method":    res.get("stattest_name") or res.get("stattest"),
                "threshold": res.get("drift_threshold") or res.get("threshold"),
            })
    return pd.DataFrame(rows)


def make_report_for_plant(
    df: pd.DataFrame,
    strategy: Literal["decay","golden","seasonal"] = BASELINE_STRATEGY,
    forced_dt_col: Optional[str] = "date_time",
    out_prefix: str = "planta",
) -> Path:

    # --- índice temporal y limpieza de outliers (igual que antes)
    dt = "date_time"
    df = df.copy()
    df[dt] = pd.to_datetime(df[dt], errors="coerce")
    df = df.dropna(subset=[dt]).sort_values(dt).set_index(dt)
    df = strip_outliers(df)

    if df.empty:
        raise ValueError("Dataset vacío tras filtrar/parsear fechas.")

    # === agregado: integrar flags si existen ==========================
    flag_path = flag_files.get(out_prefix)
    if flag_path and flag_path.exists():
        flags = pd.read_csv(flag_path, parse_dates=["date_time"])
        flags["date_time"] = pd.to_datetime(flags["date_time"]).dt.floor("min")
        df.index = df.index.floor("min")
        df = df.merge(flags, left_index=True, right_on="date_time", how="left").set_index("date_time")

        # --- filtro por columna: solo se eliminan datos faltantes de ESA variable
        nd_cols = [c for c in df.columns if c.startswith("nd_")]
        for nd_col in nd_cols:
            var = nd_col.replace("nd_", "")
            if var in df.columns:
                mask = ~df[nd_col]  # True donde hay dato válido
                df.loc[~mask, var] = np.nan  # marca como NaN solo esa columna

        # eliminar columnas de control de los flags
        drop_cols = ["valid_for_drift", "nd_any", "nd_all"] + nd_cols
        df = df[[c for c in df.columns if c not in drop_cols]]

        print(f"[{out_prefix}] Flags integrados (por columna) → {len(df)} filas totales, sin eliminar registros completos")
    else:
        print(f"[{out_prefix}] Sin flags o archivo no encontrado, se usa DF completo.")
    # ================================================================

    # --- split temporal (idéntico al tuyo)
    now = df.index.max()
    cur_start = now - pd.to_timedelta(CURRENT_WINDOW)
    cur = df.loc[cur_start:now]
    hist = df.loc[:cur_start - pd.Timedelta(nanoseconds=1)]

    # --- baseline determinístico
    if strategy == "decay":
        ref_global = ref_decay_prefix_mass(hist, now, DECAY_HALF_LIFE_HOURS, DECAY_WEIGHT_MASS)
    elif strategy == "golden":
        ref_global = ref_golden(hist, GOLDEN_WIN, GOLDEN_STEP, GOLDEN_K)
    elif strategy == "seasonal":
        ref_global = ref_seasonal(hist, now, SEASONAL_WEEKS_BACK)
    else:
        raise ValueError("strategy inválida")

    if ref_global.empty:
        ref_global = hist

    # --- columnas comunes menos dt/exclude
    common_cols = sorted(set(ref_global.columns).intersection(cur.columns) - {dt} - set(EXCLUDE_COLUMNS))
    if not common_cols:
        raise ValueError("No hay columnas comunes para comparar.")
    ref_final = ref_global[common_cols].copy()
    cur_final = cur[common_cols].copy()

    # --- RESAMPLE
    if RESAMPLE:
        ref_final = resample_mixed(ref_final, RESAMPLE, RESAMPLE_AGG).dropna(how="all")
        cur_final = resample_mixed(cur_final, RESAMPLE, RESAMPLE_AGG).dropna(how="all")

    # --- tipos y columnas 100% NaN (igual que antes)
    numeric_cols, categorical_cols, dropped_all_nan = build_types_keep_all(ref_final, cur_final, dt_col=dt)

    # --- (resto de make_report_for_plant sin tocar) -------------------
    audit_rows = []
    for c in common_cols:
        reason = []
        if c in EXCLUDE_COLUMNS: reason.append("in_EXCLUDE_COLUMNS")
        if c in dropped_all_nan: reason.append("all_nan_ref_and_cur")
        kept = (c not in dropped_all_nan) and (c not in EXCLUDE_COLUMNS)
        audit_rows.append({"col": c, "kept": kept, "reason": ";".join(reason)})
    audit_df = pd.DataFrame(audit_rows).sort_values(["kept","col"])
    tag_base = f"{strategy}_{'resamp'+RESAMPLE if RESAMPLE else 'raw'}_{pd.Timestamp(now).strftime('%Y%m%d_%H%M%S')}"

    if not numeric_cols and not categorical_cols:
        raise ValueError("Todas las columnas quedaron 100% NaN en ref y cur tras el resample.")

    definition = DataDefinition(
        numerical_columns=numeric_cols if numeric_cols else None,
        categorical_columns=categorical_cols if categorical_cols else None
    )
    preset_kwargs = {}
    if NUM_METHOD != "auto":
        preset_kwargs["num_method"] = NUM_METHOD
        if NUM_THRESHOLD is not None:
            preset_kwargs["num_threshold"] = NUM_THRESHOLD

    metrics = [
        DataDriftPreset(**preset_kwargs),
        DriftedColumnsCount(**preset_kwargs),
        *[
            (ValueDrift(column=c) if NUM_METHOD == "auto"
             else (ValueDrift(column=c, method=NUM_METHOD) if NUM_THRESHOLD is None
                   else ValueDrift(column=c, method=NUM_METHOD, threshold=NUM_THRESHOLD)))
            for c in (numeric_cols + categorical_cols)
        ],
    ]

    report = Report(metrics=metrics)
    ds_ref = Dataset.from_pandas(ref_final.reset_index(drop=True), data_definition=definition)
    ds_cur = Dataset.from_pandas(cur_final.reset_index(drop=True), data_definition=definition)
    snap = report.run(reference_data=ds_ref, current_data=ds_cur)

    # (tu código original de guardado igual)
    df_cols = extract_value_drift_table(report, snap)
    drifted_count = int(df_cols.get("drifted", pd.Series(dtype=bool)).fillna(False).sum())
    total_cols = int(df_cols.shape[0])

    kpi_tag = ""
    kpi_present = KPI_ENABLED and (KPI_COL in df_cols["col"].tolist())
    kpi_drifted = bool(df_cols.query("col == @KPI_COL and drifted == True").shape[0]) if kpi_present else False
    if KPI_ENABLED and KPI_IN_FILENAME:
        kpi_tag = "_KPI-DRIFT" if kpi_drifted else "_KPI-OK" if kpi_present else "_KPI-N/A"

    out_html = output_dir / f"{out_prefix}_{strategy}.html"
    snap.save_html(str(out_html))


    print(f"[{out_prefix}] cols={total_cols} | drifted={drifted_count} | KPI={'DRIFT' if kpi_drifted else 'OK' if kpi_present else 'N/A'}")
    print(f"OK → {out_html.name} (carpeta: {output_dir})")
    return out_html

In [8]:
for plant, path in plant_files.items():
    output_dir = output_dirs[plant]  # asigna el output_dir correspondiente
    for strategy in ["decay", "golden", "seasonal"]:
        df = pd.read_csv(path)
        make_report_for_plant(df, strategy=strategy, out_prefix=plant)

[planta1] Flags integrados (por columna) → 82286 filas totales, sin eliminar registros completos
[planta1] cols=0 | drifted=0 | KPI=N/A
OK → planta1_decay.html (carpeta: ..\reportes_D3\newnew\planta1)
[planta1] Flags integrados (por columna) → 82286 filas totales, sin eliminar registros completos
[planta1] cols=0 | drifted=0 | KPI=N/A
OK → planta1_golden.html (carpeta: ..\reportes_D3\newnew\planta1)
[planta1] Flags integrados (por columna) → 82286 filas totales, sin eliminar registros completos
[planta1] cols=0 | drifted=0 | KPI=N/A
OK → planta1_seasonal.html (carpeta: ..\reportes_D3\newnew\planta1)
[planta2] Flags integrados (por columna) → 440471 filas totales, sin eliminar registros completos
[planta2] cols=0 | drifted=0 | KPI=N/A
OK → planta2_decay.html (carpeta: ..\reportes_D3\newnew\planta2)
[planta2] Flags integrados (por columna) → 440471 filas totales, sin eliminar registros completos
[planta2] cols=0 | drifted=0 | KPI=N/A
OK → planta2_golden.html (carpeta: ..\reportes_D3\newn